# Paper 5 (RQ-F6) — Detecting coordinated algorithmic manipulation

Generates the labelled training data and the market-share feasibility sweeps for the manipulation-detection classifier.

Self-contained Colab notebook. **Runtime -> Run all**, or run cells top to bottom.
Clones the repo, installs the package, runs the scenarios this paper needs, and saves
the result CSVs to Google Drive (or Supabase).


## 1. Clone the repo and install


In [ ]:
REPO = 'https://github.com/zalliata/marketsim-v2.git'
# Private repo? Uncomment and paste a GitHub token (Contents: read):
# TOKEN = 'ghp_xxx'; REPO = REPO.replace('https://', f'https://{TOKEN}@')
import os, shutil
if os.path.exists('marketsim-v2'): shutil.rmtree('marketsim-v2')
!git clone -q $REPO
%cd marketsim-v2
!pip install -q -e .
print('installed:', os.path.exists('pyproject.toml'))


## 2. Sanity check


In [ ]:
!python -m pytest -q


## 3. (Optional) Save results to Google Drive
Skip to keep results only in the temporary Colab filesystem.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/marketsim-results', exist_ok=True)
print('Drive mounted')


## 4. (Optional) Write to Supabase instead of local CSV
Leave unrun for local CSV (default). Prefer Colab Secrets over pasting keys.


In [ ]:
import os
# os.environ['SUPABASE_URL'] = 'https://xxxx.supabase.co'
# os.environ['SUPABASE_SERVICE_KEY'] = 'your-service-key'
# !pip install -q supabase
print('Supabase configured:', bool(os.environ.get('SUPABASE_URL')))


## 5. Shared runner


In [ ]:
from marketsim.scenarios import definitions as D
from marketsim.runner.batch import run_batch, run_sweep
from marketsim.storage import make_backend

ITER = 20         # per condition; set to 100 for the reported runs
LLM  = 'scripted' # deterministic adversary (free). 'anthropic' etc. only for tiny batches

def run_scenarios(ids, iterations=None, llm=None):
    iterations = iterations or ITER; llm = llm or LLM
    backend = make_backend('auto')
    try:
        for sid in ids:
            sc = D.get(sid)
            if sc.sweep:
                stats = run_sweep(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] sweep {sc.sweep.kind} - {len(stats)} points:')
                for st in stats:
                    print(f'    {st.grid_key}={st.grid_value:<7} vol={st.vol_mean:.4f} '
                          f'adv_pnl={st.adversary_pnl_mean:>12,.0f} cascade={st.cascade_freq_mean:.3f} '
                          f'stab={st.stabilisation_mean:.3f}')
            else:
                st = run_batch(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] batch n={st.n}: vol={st.vol_mean:.4f} '
                      f'adv_pnl={st.adversary_pnl_mean:,.0f} cascade={st.cascade_freq_mean:.3f}')
    finally:
        backend.close()
    print('--- done ---')


## 6. Labelled detection battery
Nine configurations, each row tagged manipulated/clean by the scenario's label. Three
clean baselines (noise, +MM, +full defense) and six adversaries (A1-A3 volatility
maximisers, G1-G3 graph exploiters). The clean runs span the same defensive configs as
the manipulated runs, so the classifier learns *manipulation*, not the presence of defense.


In [ ]:
PAPER5_LABELLED = [
    'p5-clean-noise', 'p5-clean-mm', 'p5-clean-defense',
    'p5-manip-A1', 'p5-manip-A2', 'p5-manip-A3',
    'p5-manip-G1', 'p5-manip-G2', 'p5-manip-G3',
]
run_scenarios(PAPER5_LABELLED)      # add iterations=100 for the reported run


## 7. Detection-feasibility sweeps
Adversary market share swept 1% -> 50%, separately for an LLM adversary and a graph
exploiter. Used to find the minimum share at which detection becomes statistically viable,
and to test H2 (graph strategies detectable at lower share than pure-volatility ones).


In [ ]:
PAPER5_SWEEPS = ['p5-share-sweep-llm', 'p5-share-sweep-graph']
run_scenarios(PAPER5_SWEEPS)        # 9 share points each


## 8. Extract detection features (starter)
Turns the raw tick logs into the aggregate-observable feature vectors the classifier
trains on (order-flow imbalance, spread, depth asymmetry, cancellation rate, price impact,
cross-asset correlation). This is a starting point — extend into a full train/eval loop.


In [ ]:
import glob, pandas as pd
from marketsim.metrics.microstructure import window_features
from marketsim.types import TickMetrics

rows = []
for f in glob.glob('results/p5-*/scenario_iterations.csv'):
    df = pd.read_csv(f)
    # label is carried in the scenario_iterations table
    rows.append(df[['scenario_id','label','realized_volatility','max_volatility',
                    'cascade_frequency','bid_ask_spread_mean','adversary_pnl']])
feat = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print(feat['label'].value_counts() if len(feat) else 'run the batteries above first')
feat.head()


## Collect results
CSVs land in `results/<run_id>/`. Copy to Drive (if mounted) or zip and download.


In [ ]:
import glob, os, shutil
runs = sorted(glob.glob('results/*'))
print(len(runs), 'run folders')
if os.path.exists('/content/drive/MyDrive'):
    dest = '/content/drive/MyDrive/marketsim-results'
    for r in runs:
        shutil.copytree(r, os.path.join(dest, os.path.basename(r)), dirs_exist_ok=True)
    print('copied to', dest)
else:
    shutil.make_archive('marketsim_results', 'zip', 'results')
    from google.colab import files; files.download('marketsim_results.zip')


---
### Notes
- Full paper run: set `ITER = 100` in cell 5 and re-run cells 6-7 (~2,700 labelled sims + 
  1,800 sweep sims).
- The per-tick microstructure features for a proper classifier live in 
  `marketsim.metrics.microstructure` (window_features, cross_asset_corr_spike); the time_steps
  CSVs hold signed_ofi, spread, depth, cancellations, price_impact per tick.
- Ask Claude to build the full classifier + feasibility-curve script when you're ready.
